# Evaluate NTP (next-token-prediction) predictions

`model_step(batch)` returns a plain dict with `logits_gen`, `targets_gen`, `mask`,
`acc_dict_gen`, etc. for a *single* batch. Unlike MPM, the `head_gen` (next-token-prediction)
task is causal/autoregressive over **all** valid particles (`mask` here is `batch["part_mask"]`,
not a special masked-positions subset) — every particle position is used to predict the
*following* particle's token id. This notebook mirrors `evaluate_mpm.ipynb`, adapted for the
`head_gen` head, building the equivalent test-run output lists manually by looping over batches.

## 1. Import Required Libraries and Modules

In [2]:
import pyrootutils

pyrootutils.setup_root(__file__ if "__file__" in dir() else ".", indicator=".project-root", pythonpath=True)

import hydra
import numpy as np
import torch
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from gabbro.models.backbone_multihead import BackboneMultiHeadLightning

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## 2. Instantiate DataModule and Load a Batch

Using Hydra's `compose` API to build the same config that was used for training (so the
preprocessing / feature dict / tokenization match the checkpoint), then instantiate the
datamodule from it directly, without going through `gabbro/train.py`'s `@hydra.main`.

In [ ]:
CKPT_PATH = (
    "/home/users/w/wozniak/dev/enhancing-ntp4jets/trained/omnijet-backbone-multihead/"
    "runs/2026-07-20_17-45-08_gpu023_ConsonantalCompote/checkpoints/epoch_029_step_300000_loss_3.16025.ckpt"
)

# Load the *resolved* config that was actually used to train this checkpoint (saved next to the
# run's checkpoints directory), instead of re-composing from the current experiment yaml. This
# matters because the experiment yaml can change over time (e.g. the `feature_dict` override in
# `example_experiment_backbone_and_head.yaml` has since been changed to the extended/all-features
# set), while this specific checkpoint was trained with a different feature_dict
# (`feature_dict_kin_massless_without_cuts.yaml`). Using the checkpoint's own resolved config
# guarantees the preprocessing/feature_dict/tokenization match what the model was actually trained on.
from pathlib import Path

resolved_cfg_path = Path(CKPT_PATH).parent.parent / "config_resolved.yaml"
cfg = OmegaConf.load(resolved_cfg_path)

print(OmegaConf.to_yaml(cfg.data.dataset_kwargs_common.feature_dict))

datamodule = hydra.utils.instantiate(cfg.data)
datamodule.setup(stage="test")

test_dataloader = datamodule.test_dataloader()
batch = next(iter(test_dataloader))

# move all tensors in the batch to the target device
batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
{k: (v.shape if torch.is_tensor(v) else v) for k, v in batch.items()}

part_pt:
  multiply_by: 1
  subtract_by: 1.8
  func: signed_log
  inv_func: signed_exp
part_etarel:
  multiply_by: 3
part_phirel:
  multiply_by: 3



Parameter seed is present in both dataset_kwargs_common (value=None) and dataset_kwargs_train(value=42). Using the value from dataset_kwargs_train.
Parameter seed is present in both dataset_kwargs_common (value=None) and dataset_kwargs_val(value=1). Using the value from dataset_kwargs_val.
Parameter seed_shuffle_data is present in both dataset_kwargs_common (value=None) and dataset_kwargs_val(value=1). Using the value from dataset_kwargs_val.
Parameter seed_shuffle_data is present in both dataset_kwargs_common (value=None) and dataset_kwargs_test(value=42). Using the value from dataset_kwargs_test.
Parameter random_seed_for_per_file_shuffling is present in both dataset_kwargs_common (value=None) and dataset_kwargs_val(value=None). Using the value from dataset_kwargs_val.
shuffle_files is False. This means that the files list will not be shuffled.
shuffle_files is False. Will not shuffle files and continue with the previous order.


## 3. Instantiate Model and Load Checkpoint

Since the model was trained with `self.save_hyperparameters()`, `load_from_checkpoint` reconstructs
it (including `head_mpm`, `head_gen`, `head_class`) exactly as it was during training — no need to
pass in the config manually. Note: `head_gen` will only exist (non-`None`) if the checkpoint was
trained with `loss_term_weights.gen != 0`.

In [6]:
model = BackboneMultiHeadLightning.load_from_checkpoint(CKPT_PATH, map_location=device)
model = model.to(device)
model.eval()

print(f"head_mpm is active: {model.head_mpm is not None}")
print(f"head_gen is active: {model.head_gen is not None}")
print(f"head_class is active: {model.head_class is not None}")

if model.head_gen is None:
    raise RuntimeError(
        "This checkpoint has no trained NTP head (loss_term_weights.gen was 0 during training). "
        "Use a checkpoint trained with gen != 0 to evaluate NTP."
    )

head_mpm is active: False
head_gen is active: True
head_class is active: False


## 4. Run `model_step` Manually

This is the same function `_shared_step` calls internally, so the numbers you get here match what
`trainer.test(...)` would log — we're just calling it ourselves, one batch at a time, without a
`Trainer`.

In [7]:
with torch.no_grad():
    model_step_output = model.model_step(batch)

model_step_output.keys()

RuntimeError: mat1 and mat2 shapes cannot be multiplied (54000x14 and 3x128)

## 5. Inspect Output Dictionary Keys and Shapes

The NTP-relevant keys are: `logits_gen` (B, T, vocab_size), `targets_gen` (B, T), `mask` (B, T) —
the valid-particle mask (`batch["part_mask"]`), since NTP predicts the next token for *every*
valid particle position, not just masked-out ones — and `acc_dict_gen` (accuracy already computed
by `calc_acc_from_logits` for a fixed set of token positions).

In [ ]:
for key in ["logits_gen", "targets_gen", "mask", "loss_gen"]:
    value = model_step_output[key]
    if torch.is_tensor(value):
        print(f"{key:35s} shape={tuple(value.shape)} dtype={value.dtype}")
    else:
        print(f"{key:35s} {value}")

print()
print("acc_dict_gen:")
for key, value in model_step_output["acc_dict_gen"].items():
    print(f"  {key}: {value}")

### 5b. Sanity-check `acc_token_i`: sample counts and top-5 accuracy

Unlike MPM, position 0 here is **not** a trivial start-token target — `targets_gen` is the
*next* particle's token id, so even position 0 requires actually predicting the first real
particle. `acc_token_i` for large `i` (e.g. 90) is still averaged over very few valid particles
(most jets don't have 90+ particles), so it can be noisy.

Below we check: (1) how many valid positions each `acc_token_i` is actually based on, and
(2) top-5 accuracy (is the true next token among the model's 5 most likely predictions?), which
is a softer/more informative metric than exact top-1 match for a large, unordered codebook.

In [ ]:
logits_gen = model_step_output["logits_gen"]
targets_gen = model_step_output["targets_gen"]
mask = model_step_output["mask"]

max_len_in_batch = logits_gen.size(1)
token_indices_to_calc = [i for i in [0, 1, 2, 3, 10, 20, 30, 40, 50, 90] if i < max_len_in_batch]

print(f"{'position':>10s}  {'n_valid_samples':>16s}  {'top1_acc':>9s}  {'top5_acc':>9s}")
for i in token_indices_to_calc:
    n_samples = mask[:, i].sum().item()

    if n_samples == 0:
        print(f"{i:>10d}  {n_samples:>16d}  {'n/a':>9s}  {'n/a':>9s}")
        continue

    logits_i = logits_gen[:, i, :]
    targets_i = targets_gen[:, i]
    mask_i = mask[:, i].bool()

    top1_correct = (torch.argmax(logits_i, dim=-1) == targets_i) & mask_i
    top1_acc = top1_correct.sum().item() / n_samples

    top5_preds = torch.topk(logits_i, k=5, dim=-1).indices  # (B, 5)
    top5_correct = (top5_preds == targets_i.unsqueeze(-1)).any(dim=-1) & mask_i
    top5_acc = top5_correct.sum().item() / n_samples

    print(f"{i:>10d}  {n_samples:>16d}  {top1_acc:>9.4f}  {top5_acc:>9.4f}")

# overall (all positions) top-5 accuracy, for comparison with acc_all_tokens (top-1)
mask_bool_all = mask.bool()
top5_preds_all = torch.topk(logits_gen, k=5, dim=-1).indices  # (B, T, 5)
top5_correct_all = (top5_preds_all == targets_gen.unsqueeze(-1)).any(dim=-1) & mask_bool_all
top5_acc_all = top5_correct_all.sum().item() / mask_bool_all.sum().item()
print(f"\nOverall top-5 accuracy (all valid positions): {top5_acc_all:.4f}")
print(f"(compare to acc_all_tokens / top-1: {model_step_output['acc_dict_gen']['acc_all_tokens']:.4f})")

## 6. Extract Predictions, Targets, and Mask Lists

Now we loop over multiple batches ourselves and build the equivalent of
`test_ntp_pred_list` / `test_ntp_target_list` / `test_ntp_mask_list` manually — mirroring what
`_collect_batch_data` would have accumulated during a real `trainer.test(...)` run.

Set `N_BATCHES` to how many batches you want to evaluate on.

In [ ]:
N_BATCHES = 20

test_ntp_pred_list = []
test_ntp_target_list = []
test_ntp_mask_list = []

test_iter = iter(datamodule.test_dataloader())

with torch.no_grad():
    for i in range(N_BATCHES):
        try:
            b = next(test_iter)
        except StopIteration:
            print(f"Test dataloader exhausted after {i} batches.")
            break

        b = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}
        out = model.model_step(b)

        # argmax token predictions (see section 7 for the sampling variant used at generation time)
        preds = torch.argmax(out["logits_gen"], dim=-1)

        test_ntp_pred_list.append(preds.cpu().numpy())
        test_ntp_target_list.append(out["targets_gen"].cpu().numpy())
        test_ntp_mask_list.append(out["mask"].cpu().numpy())

print(f"Collected {len(test_ntp_pred_list)} batches.")
print(f"Example batch shapes: preds={test_ntp_pred_list[0].shape}, "
      f"targets={test_ntp_target_list[0].shape}, "
      f"mask={test_ntp_mask_list[0].shape}")

## 7. Convert Logits to Token Predictions

We already used `argmax` above for simplicity. The model's own `generate_batch_continuous`
instead *samples* from the softmax distribution and forbids token id `0` (reserved as the
start/stop token) — replicate that here if you want predictions consistent with generation-time
behavior rather than a plain argmax.

In [ ]:
def sample_gen_predictions(logits_gen: torch.Tensor) -> torch.Tensor:
    """Sample predicted token ids from logits, excluding token id 0 (start/stop token),
    mirroring `generate_batch_continuous` in gabbro/models/backbone_multihead.py.
    """
    probs = torch.softmax(logits_gen[..., 1:], dim=-1)  # exclude token id 0
    b, t, c = probs.shape
    sampled = torch.multinomial(probs.reshape(b * t, c), 1).reshape(b, t)
    return sampled + 1  # shift back since we excluded index 0


with torch.no_grad():
    sampled_preds = sample_gen_predictions(model_step_output["logits_gen"])

argmax_preds = torch.argmax(model_step_output["logits_gen"], dim=-1)

print(f"argmax preds shape: {argmax_preds.shape}")
print(f"sampled preds shape: {sampled_preds.shape}")

## 8. Align Predictions with Targets Using the Valid-Particle Mask

Only positions where `mask == 1` correspond to actual valid particles — everything else
(padding beyond the jet's real particle count) should be excluded before computing
accuracy/confusion matrices.

In [ ]:
all_preds_masked = []
all_targets_masked = []

for preds, targets, mask in zip(test_ntp_pred_list, test_ntp_target_list, test_ntp_mask_list):
    mask_bool = mask.astype(bool)
    all_preds_masked.append(preds[mask_bool])
    all_targets_masked.append(targets[mask_bool])

all_preds_masked = np.concatenate(all_preds_masked)
all_targets_masked = np.concatenate(all_targets_masked)

print(f"Number of valid-particle predictions collected: {len(all_preds_masked)}")

## 9. Compute Accuracy and Confusion Matrix

In [ ]:
accuracy = (all_preds_masked == all_targets_masked).mean()
print(f"Overall NTP accuracy on valid particle positions: {accuracy:.4f}")

# limit the confusion matrix to the most frequent tokens for readability
top_k = 20
unique, counts = np.unique(all_targets_masked, return_counts=True)
top_tokens = unique[np.argsort(-counts)][:top_k]

mask_top = np.isin(all_targets_masked, top_tokens) & np.isin(all_preds_masked, top_tokens)
cm = confusion_matrix(
    all_targets_masked[mask_top],
    all_preds_masked[mask_top],
    labels=top_tokens,
    normalize="true",
)

fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=top_tokens).plot(
    ax=ax, xticks_rotation=90, colorbar=True, cmap="viridis"
)
ax.set_title(f"NTP confusion matrix (top {top_k} tokens, row-normalized)")
plt.tight_layout()
plt.show()

### 9b. Check for start/stop-token collapse in NTP predictions

Unlike MPM, no position here is guaranteed to have a trivial start-token target (`targets_gen`
is always the *next* real particle's token id), so any bias toward predicting token `0` (the
reserved start/stop id) as the next token would be a genuine failure mode worth flagging.

In [ ]:
frac_pred_is_start_token = (all_preds_masked == 0).mean()
frac_target_is_start_token = (all_targets_masked == 0).mean()

print(f"fraction of predictions == token 0 (start/stop): {frac_pred_is_start_token:.4f}")
print(f"fraction of TRUE targets == token 0:              {frac_target_is_start_token:.4f}")

pred_unique, pred_counts = np.unique(all_preds_masked, return_counts=True)
top_predicted = pred_unique[np.argsort(-pred_counts)][:10]
top_predicted_frac = pred_counts[np.argsort(-pred_counts)][:10] / len(all_preds_masked)

print("\nTop 10 most frequently *predicted* tokens:")
for tok, frac in zip(top_predicted, top_predicted_frac):
    print(f"  token {tok:>6d}: {frac:.4f} of all predictions")

## 10. Visualize Sample Predictions vs Ground Truth

Pick a single example (jet) from the last collected batch and plot its sequence of predicted
vs. true next-particle tokens at the valid particle positions.

In [ ]:
example_idx = 0  # index within the last collected batch

preds = test_ntp_pred_list[-1][example_idx]
targets = test_ntp_target_list[-1][example_idx]
mask = test_ntp_mask_list[-1][example_idx].astype(bool)

positions = np.arange(len(preds))[mask]
preds_masked = preds[mask]
targets_masked = targets[mask]

fig, ax = plt.subplots(figsize=(12, 4))
width = 0.4
ax.bar(positions - width / 2, targets_masked, width=width, label="true next token", alpha=0.8)
ax.bar(positions + width / 2, preds_masked, width=width, label="predicted next token", alpha=0.8)
ax.set_xlabel("Particle position in sequence")
ax.set_ylabel("Token id")
ax.set_title(f"NTP predictions vs. ground truth (example {example_idx} in last batch)")
ax.legend()
plt.tight_layout()
plt.show()

match = (preds_masked == targets_masked)
print(f"Correct: {match.sum()} / {len(match)} valid positions ({match.mean():.2%})")

## 11. Compare `test_loss_gen` / accuracy Across Checkpoints

Load every saved checkpoint in the run directory (in epoch order) and evaluate NTP loss/accuracy
on the *same* fixed set of test batches, to see whether NTP performance is still improving or has
plateaued.

This can be slow (reloads the model + reruns forward passes per checkpoint) — reduce
`N_EVAL_BATCHES` if needed.

In [ ]:
import re
from pathlib import Path

CHECKPOINT_DIR = Path(CKPT_PATH).parent
N_EVAL_BATCHES = 5  # number of fixed batches to evaluate each checkpoint on

# collect "epoch_XXX_step_YYY_loss_ZZZ.ckpt" checkpoints (skip best.ckpt / last.ckpt, no epoch info)
ckpt_paths = sorted(
    CHECKPOINT_DIR.glob("epoch_*.ckpt"),
    key=lambda p: int(re.search(r"epoch_(\d+)_", p.name).group(1)),
)
print(f"Found {len(ckpt_paths)} epoch checkpoints in {CHECKPOINT_DIR}")
for p in ckpt_paths:
    print(f"  {p.name}")

In [ ]:
import gc

# free the original `model` from GPU memory first — otherwise it sits alongside each
# freshly-loaded `ckpt_model` below and doubles the VRAM footprint, causing OOM
model = model.to("cpu")
gc.collect()
torch.cuda.empty_cache()

# the dataloader's batch size can be too large to run a forward pass on for every checkpoint
# without OOM-ing, so instead of shrinking N_EVAL_BATCHES, split each fixed batch into smaller
# sub-batches and evaluate those sequentially, accumulating a sample-weighted average — this
# keeps every collected sample without needing all of it on the GPU at once.
EVAL_SUB_BATCH_SIZE = 32

# pre-fetch a fixed set of batches once (kept on CPU), so every checkpoint is evaluated on
# identical data
fixed_batches = []
_iter = iter(datamodule.test_dataloader())
for _ in range(N_EVAL_BATCHES):
    b = next(_iter)
    b = {k: (v.cpu() if torch.is_tensor(v) else v) for k, v in b.items()}
    fixed_batches.append(b)


def iter_sub_batches(batch, sub_batch_size):
    """Yield successive sub-batches of a batch dict, moved to `device`."""
    n = next(v.shape[0] for v in batch.values() if torch.is_tensor(v))
    for start in range(0, n, sub_batch_size):
        stop = start + sub_batch_size
        yield {
            k: (v[start:stop].to(device) if torch.is_tensor(v) else v) for k, v in batch.items()
        }, stop - start if stop <= n else n - start


results = []
for ckpt_path in ckpt_paths:
    epoch = int(re.search(r"epoch_(\d+)_", ckpt_path.name).group(1))
    ckpt_model = BackboneMultiHeadLightning.load_from_checkpoint(str(ckpt_path), map_location=device)
    ckpt_model = ckpt_model.to(device).eval()

    total_samples = 0
    weighted_loss_sum = 0.0
    weighted_acc_sum = 0.0
    with torch.no_grad():
        for b in fixed_batches:
            for sub_batch, n_samples in iter_sub_batches(b, EVAL_SUB_BATCH_SIZE):
                out = ckpt_model.model_step(sub_batch)
                weighted_loss_sum += out["loss_gen"].item() * n_samples
                weighted_acc_sum += out["acc_dict_gen"]["acc_all_tokens"].item() * n_samples
                total_samples += n_samples
                del out, sub_batch

    results.append(
        {
            "epoch": epoch,
            "ckpt": ckpt_path.name,
            "loss_gen": weighted_loss_sum / total_samples,
            "acc_all_tokens": weighted_acc_sum / total_samples,
        }
    )
    print(f"epoch {epoch:>4d}: loss_gen={results[-1]['loss_gen']:.4f}  acc_all_tokens={results[-1]['acc_all_tokens']:.4f}")

    ckpt_model = ckpt_model.to("cpu")
    del ckpt_model
    gc.collect()
    torch.cuda.empty_cache()

# restore the original model back onto the GPU for any later cells that still use it
model = model.to(device)

In [ ]:
epochs = [r["epoch"] for r in results]
losses = [r["loss_gen"] for r in results]
accs = [r["acc_all_tokens"] for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, losses, marker="o")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("test loss_gen")
ax1.set_title("NTP loss vs. epoch")

ax2.plot(epochs, accs, marker="o", color="tab:orange")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("acc_all_tokens (top-1)")
ax2.set_title("NTP top-1 accuracy vs. epoch")

plt.tight_layout()
plt.show()